In [133]:
import pandas as pd
import numpy as np
import duckdb

# Visualisation
import plotly.express as px
import plotly.graph_objects as go

### Import files


In [134]:
# Connexion à la base de données DuckDB
con = duckdb.connect("dev.duckdb", read_only=True)

#### Budget des communes


In [135]:
# Récupération de la table
table_name = "budget_per_compte_communes"
compte_table = con.sql(f"SELECT * FROM {table_name}")

budget = compte_table.df()
budget

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,annee,siret,code_departement,code_insee,code_region,type_compte,solde_debiteur,solde_crediteur,solde,code_geo
0,2010,21010001200017,01,1,82,depenses,329891.87,5999.38,323892.49,011
1,2010,21010010300014,01,10,82,depenses,1160202.00,590.23,1159611.77,0110
2,2010,21010100200017,01,100,82,depenses,101188.40,0.00,101188.40,01100
3,2010,21010101000010,01,101,82,depenses,138281.77,0.00,138281.77,01101
4,2010,21010102800012,01,102,82,depenses,344885.95,1636.00,343249.95,01102
...,...,...,...,...,...,...,...,...,...,...
2691321,2024,20000882900018,106,613,106,primes d assurances,26772.25,0.00,26772.25,106613
2691322,2024,20000885200010,106,614,106,primes d assurances,30557.30,0.00,30557.30,106614
2691323,2024,20000886000013,106,615,106,primes d assurances,59865.34,0.00,59865.34,106615
2691324,2024,20000887800015,106,616,106,primes d assurances,96251.39,0.00,96251.39,106616


#### SIREN Insee


In [136]:
insee_siren = pd.read_csv(
    "C:/Users/lesli/kDrive2/Data/Data For Good/14_prixchangementclimatique/Sources/banatic-sireninsee-last.csv",
    sep=";",
    encoding="latin-1",
)
insee_siren.dtypes

Reg_com       int64
dep_com      object
siren         int64
insee        object
nom_com      object
ptot_2024    object
pmun_2024    object
pcap_2024    object
dtype: object

In [137]:
# Conversion de la colonne siren en string
insee_siren["siren"] = insee_siren["siren"].astype("str")

# Vérification du nombre de catactères
insee_siren["len"] = insee_siren["siren"].str.len()
insee_siren.groupby(["len"])["insee"].nunique()

len
9    34935
Name: insee, dtype: int64

In [138]:
# Création d'une table avec un seul code siren
insee_siren = insee_siren[["siren", "insee", "nom_com"]].copy()
insee_siren.drop_duplicates(subset="siren", inplace=True)
insee_siren

,siren,insee,nom_com
0,210100012,01001,L'Abergement-Clémenciat
1,210100020,01002,L'Abergement-de-Varey
2,210100046,01004,Ambérieu-en-Bugey
3,210100053,01005,Ambérieux-en-Dombes
4,210100061,01006,Ambléon
...,...,...,...
34930,200008829,97613,M'Tsangamouji
34931,200008852,97614,Ouangani
34932,200008860,97615,Pamandzi
34933,200008878,97616,Sada


#### Liste des catastrophes naturelles


In [139]:
table_name = "dev.main.ccr_details"
catnat_table = con.sql(f"SELECT * FROM {table_name}")

catnat = catnat_table.df()
catnat.head(5)

,code_geo,nom_commune,date_debut_evenement,date_fin_evenement,date_arrete,date_parution_jo,nom_peril,franchise,libelle_avis,code_arrete
0,07005,ALBA LA ROMAINE,2025-11-16,2025-11-16,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,-,Non reconnue,904
1,07022,BAIX,2025-11-16,2025-11-17,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,Simple,Reconnue,904
2,07181,LE POUZIN,2025-11-16,2025-11-16,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,-,Non reconnue,904
3,07198,ROMPON,2025-11-15,2025-11-16,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,-,Non reconnue,904
4,07219,SAINT BAUZILE,2025-11-16,2025-11-16,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,Simple,Reconnue,904


#### Population par commune


In [140]:
table_name = "dev.population_code_geo"
pop_table = con.sql(f"SELECT * FROM {table_name}")

pop = pop_table.df()
pop.head(5)

,code_geo,nom_geo,code_departement,code_region,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,pop_2025,pop_2026
0,45277,Saint-Florent-le-Jeune,45,24,447,444,448,453,457,455,453,451,448,445,442
1,97230,La Trinité,972,02,13253,12973,12771,12512,12243,12232,12025,11860,11747,11622,11454
2,97219,Le Prêcheur,972,02,1632,1541,1449,1357,1304,1252,1203,1291,1377,1463,1479
3,97208,Fonds-Saint-Denis,972,02,813,802,790,761,730,700,680,661,641,641,640
4,97225,Saint-Pierre,972,02,4285,4229,4177,4125,4123,4122,4121,4107,4088,4069,3961


### Merge des fichiers


#### Budget et INSEE pour ne garder que les communes


In [141]:
# Création de la colonne siren qui correspond aux 9 premiers caractères de la colonne siret en format string
budget["siren"] = budget["siret"].astype("str").str[:9]

In [142]:
# Merge du fichier du budget avec la liste des communes
budget = pd.merge(
    budget, insee_siren[["siren", "insee", "nom_com"]], on="siren", how="left"
)

In [143]:
# Suppression des lignes ne correspondant pas à une commune (nom_com est vide)
budget = budget.loc[budget["nom_com"] != ""]

#### Ajout de la population


In [144]:
budget = pd.merge(
    budget,
    pop[["code_geo", "pop_2024"]],
    left_on="insee",
    right_on="code_geo",
    how="left",
)

#### Ajout du nombre de catnat par commune


In [ ]:
catnat_group = catnat.copy()

# Extract year from the end catnat date
catnat_group["catnat_annee"] = pd.DatetimeIndex(catnat_group["date_fin_evenement"]).year

In [146]:
# Nombre de cat nat par commune
catnat_tot = (
    catnat.groupby(["code_geo"], as_index=False)["date_debut_evenement"]
    .nunique()
    .copy()
)
catnat_tot.rename(columns={"date_debut_evenement": "catnat_count"}, inplace=True)

In [ ]:
# Date de la cat nat par année par commune
catnat_group = (
    catnat_group.groupby(["code_geo", "catnat_annee"], as_index=False)[
        "date_debut_evenement"
    ]
    .nunique()
    .copy()
)

# Ajout d'une colonne combinant l'année et le code geo
catnat_group["annee_insee"] = (
    catnat_group["catnat_annee"].astype(str) + catnat_group["code_geo"]
)
catnat_group.head(5)

,code_geo,catnat_annee,date_debut_evenement,annee_insee
0,01001,1984,1,198401001
1,01001,2009,1,200901001
2,01002,1990,1,199001002
3,01002,2000,1,200001002
4,01004,1983,1,198301004


#### Ajout d'une clé unique


In [148]:
# Création d'une clé unique année + code insee
budget["annee_insee"] = budget["annee"].astype(str) + budget["insee"]
budget.head(5)

,annee,siret,code_departement,code_insee,code_region,type_compte,solde_debiteur,solde_crediteur,solde,code_geo_x,siren,insee,nom_com,code_geo_y,pop_2024,annee_insee
0,2010,21010001200017,01,1,82,depenses,329891.87,5999.38,323892.49,011,210100012,01001,L'Abergement-Clémenciat,01001,832,201001001
1,2010,21010010300014,01,10,82,depenses,1160202.00,590.23,1159611.77,0110,210100103,01010,Anglefort,01010,1125,201001010
2,2010,21010100200017,01,100,82,depenses,101188.40,0.00,101188.40,01100,210101002,01100,Cheignieu-la-Balme,01100,131,201001100
3,2010,21010101000010,01,101,82,depenses,138281.77,0.00,138281.77,01101,210101010,01101,Chevillard,01101,150,201001101
4,2010,21010102800012,01,102,82,depenses,344885.95,1636.00,343249.95,01102,210101028,01102,Chevroux,01102,973,201001102


In [ ]:
# Merge du fichier du budget avec le nombre de catnat par commune
budget = pd.merge(
    budget, catnat_group[["annee_insee", "catnat_annee"]], on="annee_insee", how="left"
)

# Merge du fichier du budget avec la date des cat nat
budget = pd.merge(budget, catnat_tot, left_on="insee", right_on="code_geo", how="left")

# Remplacement des valeurs NaN en 0
budget["catnat_count"] = np.nan_to_num(budget["catnat_count"])

# Suppression des colonnes code geo
budget.drop(columns=["code_geo", "code_geo_x", "code_geo_y"], inplace=True)

budget.head(5)

,annee,siret,code_departement,code_insee,code_region,type_compte,solde_debiteur,solde_crediteur,solde,siren,insee,nom_com,pop_2024,annee_insee,catnat_annee,catnat_count
0,2010,21010001200017,01,1,82,depenses,329891.87,5999.38,323892.49,210100012,01001,L'Abergement-Clémenciat,832,201001001,NaN,2.0
1,2010,21010010300014,01,10,82,depenses,1160202.00,590.23,1159611.77,210100103,01010,Anglefort,1125,201001010,NaN,0.0
2,2010,21010100200017,01,100,82,depenses,101188.40,0.00,101188.40,210101002,01100,Cheignieu-la-Balme,131,201001100,NaN,1.0
3,2010,21010101000010,01,101,82,depenses,138281.77,0.00,138281.77,210101010,01101,Chevillard,150,201001101,NaN,2.0
4,2010,21010102800012,01,102,82,depenses,344885.95,1636.00,343249.95,210101028,01102,Chevroux,973,201001102,NaN,3.0


### Préparation des données


In [150]:
# Ajout d'une colonne prime basé sur les lignes "primes d asssurances" de la colonne type de compte
budget["prime"] = np.where(
    budget["type_compte"] == "primes d assurances", budget["solde_debiteur"], 0
)

#### Groupement sur le budget total


In [ ]:
# Regroupement des données pour n'avoir qu'une ligne par commune et par année
budget_tot = budget.groupby(
    [
        "annee_insee",
        "annee",
        "insee",
        "nom_com",
        "pop_2024",
        "catnat_annee",
        "catnat_count",
    ],
    as_index=False,
).agg(budget=("solde_debiteur", "sum"), prime=("prime", "sum"))
budget_tot.head(5)

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime
0,201001159,2010,01159,Feillens,3381,2010.0,12.0,1773327.21,20060.21
1,201001202,2010,01202,Lagnieu,7268,2010.0,9.0,4225284.32,24839.94
2,201001305,2010,01305,Pont-de-Vaux,2189,2010.0,14.0,2830554.77,64141.16
3,201001323,2010,01323,Reyssouze,1010,2010.0,6.0,270689.12,3065.34
4,201001337,2010,01337,Saint-Bénigne,1343,2010.0,14.0,354212.92,6979.74


In [152]:
# Renommage de la colonne solde debiteur
budget_tot.rename(columns={"solde_debiteur": "budget"}, inplace=True)

In [153]:
# Vérification du nombre de communes par année
budget_tot.groupby("annee")["insee"].nunique()

annee
2010     2779
2011     3616
2012     2241
2013     2335
2014     2765
2015     2831
2016     5412
2017     3362
2018     9016
2019     7304
2020     6435
2021     4332
2022    10286
2023     6146
2024     5281
Name: insee, dtype: int64

#### Vérification des doublons


In [154]:
# Vérification qu'il n'y ait pas de doublons
duplicates = budget_tot[budget_tot.duplicated(subset=["annee_insee"], keep=False)]
duplicates

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime


#### Calcul de la part de la prime dans le budget total


In [155]:
# Remplacement des valeurs NaN en 0
budget_tot["budget"] = np.nan_to_num(budget_tot["budget"])
budget_tot["prime"] = np.nan_to_num(budget_tot["prime"])

In [156]:
# Vérification du nombre de lignes sans prime
budget_tot.loc[budget_tot["prime"] == 0]

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime
105,201007238,2010,07238,Saint-Genest-de-Beauzon,338,2010.0,5.0,87488.23,0.0
1109,201033143,2010,33143,Cubzac-les-Ponts,2581,2010.0,24.0,963974.78,0.0
1478,201050429,2010,50429,Regnéville-sur-Mer,733,2010.0,6.0,477626.41,0.0
1500,201050594,2010,50594,Teurthéville-Hague,1032,2010.0,3.0,390143.03,0.0
2072,201081126,2010,81126,Lacougotte-Cadoul,180,2010.0,9.0,54583.03,0.0
...,...,...,...,...,...,...,...,...,...
73464,202482005,2024,82005,Aucamville,1541,2024.0,24.0,1091379.77,0.0
73475,202482057,2024,82057,Fabas,688,2024.0,18.0,502745.22,0.0
74117,202497401,2024,97401,Les Avirons,11434,2024.0,17.0,17857187.75,0.0
74130,202497414,2024,97414,Saint-Louis,53935,2024.0,20.0,92411546.00,0.0


In [157]:
# Calcul de la part de la prime dans le budget total
budget_tot["part_prime"] = budget_tot["prime"] / budget_tot["budget"]
budget_tot.head(5)

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime,part_prime
0,201001159,2010,01159,Feillens,3381,2010.0,12.0,1773327.21,20060.21,0.011312
1,201001202,2010,01202,Lagnieu,7268,2010.0,9.0,4225284.32,24839.94,0.005879
2,201001305,2010,01305,Pont-de-Vaux,2189,2010.0,14.0,2830554.77,64141.16,0.022660
3,201001323,2010,01323,Reyssouze,1010,2010.0,6.0,270689.12,3065.34,0.011324
4,201001337,2010,01337,Saint-Bénigne,1343,2010.0,14.0,354212.92,6979.74,0.019705


#### Calcul du montant de la prime par habitant


In [158]:
# Conversion de la colonne pop_2024 en float
budget_tot["pop_2024"] = budget_tot["pop_2024"].astype("float")

In [159]:
# Remplacement des valeurs NaN en 0
budget_tot["pop_2024"] = np.nan_to_num(budget_tot["pop_2024"])

# Calcul du montant de la prime par habitant si population 2024 > 0
budget_tot["prime_per_hab"] = np.where(
    budget_tot["pop_2024"] > 0, budget_tot["prime"] / budget_tot["pop_2024"], 0
)
budget_tot.head(5)

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime,part_prime,prime_per_hab
0,201001159,2010,01159,Feillens,3381.0,2010.0,12.0,1773327.21,20060.21,0.011312,5.933218
1,201001202,2010,01202,Lagnieu,7268.0,2010.0,9.0,4225284.32,24839.94,0.005879,3.417713
2,201001305,2010,01305,Pont-de-Vaux,2189.0,2010.0,14.0,2830554.77,64141.16,0.022660,29.301581
3,201001323,2010,01323,Reyssouze,1010.0,2010.0,6.0,270689.12,3065.34,0.011324,3.034990
4,201001337,2010,01337,Saint-Bénigne,1343.0,2010.0,14.0,354212.92,6979.74,0.019705,5.197126


#### Regroupement des montants par tranches


##### Part de la prime dans le budget


In [160]:
# Conditions
conditions = [
    (budget_tot["part_prime"] == 0),
    (budget_tot["part_prime"] <= 0.01),
    (budget_tot["part_prime"] <= 0.02),
    (budget_tot["part_prime"] <= 0.05),
    (budget_tot["part_prime"] > 0.05),
]

# Catégories
results = [
    "0. Pas de prime",
    "1. 0-1%",
    "2. 1-2%",
    "3. 2-5%",
    "4. >5%",
]

# Création d'une colonne prime_cat à partir des conditions
budget_tot["prime_cat"] = np.select(conditions, results)

# Regroupement par catégorie
budget_tot.groupby(["prime_cat"]).count()

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime,part_prime,prime_per_hab
prime_cat,,,,,,,,,,,
0. Pas de prime,881,881,881,881,881,881,881,881,881,881,881
1. 0-1%,23120,23120,23120,23120,23120,23120,23120,23120,23120,23120,23120
2. 1-2%,29122,29122,29122,29122,29122,29122,29122,29122,29122,29122,29122
3. 2-5%,20274,20274,20274,20274,20274,20274,20274,20274,20274,20274,20274
4. >5%,744,744,744,744,744,744,744,744,744,744,744


##### Tranches de montant du budget


In [161]:
# Conditions
conditions = [
    (budget_tot["budget"] < 100000),
    (budget_tot["budget"] < 200000),
    (budget_tot["budget"] < 500000),
    (budget_tot["budget"] < 1000000),
    (budget_tot["budget"] >= 1000000),
]

# Catégories
results = [
    "1. 0 - 100K",
    "2. 100K-200K",
    "3. 200K-500K",
    "4. 500K-1M",
    "5.>1M",
]

# Création d'une colonne pour budget_cat à partir des conditions
budget_tot["budget_cat"] = np.select(conditions, results)

# Regroupement par catégories
budget_tot.groupby(["budget_cat"]).count()

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime,part_prime,prime_per_hab,prime_cat
budget_cat,,,,,,,,,,,,
1. 0 - 100K,5216,5216,5216,5216,5216,5216,5216,5216,5216,5216,5216,5216
2. 100K-200K,9805,9805,9805,9805,9805,9805,9805,9805,9805,9805,9805,9805
3. 200K-500K,16986,16986,16986,16986,16986,16986,16986,16986,16986,16986,16986,16986
4. 500K-1M,12769,12769,12769,12769,12769,12769,12769,12769,12769,12769,12769,12769
5.>1M,29365,29365,29365,29365,29365,29365,29365,29365,29365,29365,29365,29365


##### Nombre de catnats


In [ ]:
# Conditions
conditions = [
    (budget_tot["catnat_count"] == 0),
    (budget_tot["catnat_count"] == 1),
    (budget_tot["catnat_count"] <= 4),
    (budget_tot["catnat_count"] <= 9),
    (budget_tot["catnat_count"] >= 10),
]

# Catégories
results = [
    "Aucune",
    "1",
    "2 - 4",
    "5 - 9",
    "10+",
]

# Création d'une colonne pour budget_cat à partir des conditions
budget_tot["catnat_group"] = np.select(conditions, results)

# Regroupement par catégories
budget_tot.groupby(["catnat_group"]).count()

,annee_insee,annee,insee,nom_com,pop_2024,catnat_annee,catnat_count,budget,prime,part_prime,prime_per_hab,prime_cat,budget_cat
catnat_group,,,,,,,,,,,,,
1,928,928,928,928,928,928,928,928,928,928,928,928,928
10+,40349,40349,40349,40349,40349,40349,40349,40349,40349,40349,40349,40349,40349
2 - 4,9649,9649,9649,9649,9649,9649,9649,9649,9649,9649,9649,9649,9649
5 - 9,23215,23215,23215,23215,23215,23215,23215,23215,23215,23215,23215,23215,23215


##### Prime par habitant


In [163]:
# Conditions
conditions = [
    (budget_tot["prime_per_hab"] < 10),
    (budget_tot["prime_per_hab"] < 50),
    (budget_tot["prime_per_hab"] < 100),
    (budget_tot["prime_per_hab"] < 200),
    (budget_tot["prime_per_hab"] >= 200),
]

# Catégories
results = [
    "1. 0 - 10€",
    "2. 10 - 50€",
    "3. 50 - 100€",
    "4. 100 - 200€",
    "5. >200€",
]

# Création d'une colonne pour pop_per_hab_cat à partir des conditions
budget_tot["pop_per_hab_cat"] = np.select(conditions, results)

# Regroupement par catégories
budget_tot.loc[budget_tot["annee"] == 2024].groupby(["pop_per_hab_cat"])[
    "insee"
].count()

pop_per_hab_cat
1. 0 - 10€       1830
2. 10 - 50€      3304
3. 50 - 100€      118
4. 100 - 200€      28
5. >200€            1
Name: insee, dtype: int64

#### Création d'un dataframe avec une seule ligne par commune


##### Pivot des données


In [164]:
# Création d'un pivot table
budget_pivot = budget_tot.pivot_table(
    values=["budget", "prime", "part_prime"],
    index=["insee", "nom_com", "catnat_count"],
    columns="annee",
    aggfunc="first",
)

budget_pivot.head(3)

budget                                 \
annee                                  2010 2011 2012 2013 2014       2015   
insee nom_com           catnat_count                                         
01004 Ambérieu-en-Bugey 12.0            NaN  NaN  NaN  NaN  NaN        NaN   
01007 Ambronay          12.0            NaN  NaN  NaN  NaN  NaN  2315311.5   
01012 Aranc             3.0             NaN  NaN  NaN  NaN  NaN        NaN   

                                                                        ...  \
annee                                2016 2017         2018       2019  ...   
insee nom_com           catnat_count                                    ...   
01004 Ambérieu-en-Bugey 12.0          NaN  NaN  16923115.42        NaN  ...   
01007 Ambronay          12.0          NaN  NaN   1484930.73        NaN  ...   
01012 Aranc             3.0           NaN  NaN    326097.72  234511.35  ...   

                                         prime                               \
annee                                     2015 2016 2017      2018     2019   
insee nom_com           catnat_count                                          
01004 Ambérieu-en-Bugey 12.0               NaN  NaN  NaN  62894.32      NaN   
01007 Ambronay          12.0          72673.36  NaN  NaN  20060.92      NaN   
01012 Aranc             3.0                NaN  NaN  NaN   7560.10  7667.45   

                                                                               \
annee                                     2020      2021      2022       2023   
insee nom_com           catnat_count                                            
01004 Ambérieu-en-Bugey 12.0          50126.96  84107.74  90963.56  112284.45   
01007 Ambronay          12.0          21565.17  22444.04  23200.87   27830.23   
01012 Aranc             3.0                NaN       NaN   7258.11        NaN   

                                                 
annee                                      2024  
insee nom_com           catnat_count             
01004 Ambérieu-en-Bugey 12.0          103667.10  
01007 Ambronay          12.0           29453.43  
01012 Aranc             3.0                 NaN  

[3 rows x 45 columns]

In [165]:
# Tranformation en dataframe
budget_pivot = pd.DataFrame(budget_pivot.to_records())
budget_pivot.head(5)

,insee,nom_com,catnat_count,"('budget', 2010)","('budget', 2011)","('budget', 2012)","('budget', 2013)","('budget', 2014)","('budget', 2015)","('budget', 2016)",...,"('prime', 2015)","('prime', 2016)","('prime', 2017)","('prime', 2018)","('prime', 2019)","('prime', 2020)","('prime', 2021)","('prime', 2022)","('prime', 2023)","('prime', 2024)"
0,01004,Ambérieu-en-Bugey,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,62894.32,NaN,50126.96,84107.74,90963.56,112284.45,103667.10
1,01007,Ambronay,12.0,NaN,NaN,NaN,NaN,NaN,2315311.5,NaN,...,72673.36,NaN,NaN,20060.92,NaN,21565.17,22444.04,23200.87,27830.23,29453.43
2,01012,Aranc,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,7560.10,7667.45,NaN,NaN,7258.11,NaN,NaN
3,01013,Arandas,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,8827.18,NaN,NaN,NaN,NaN,NaN,NaN
4,01014,Arbent,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41924.22,41881.74,NaN


##### Renommage


In [ ]:
# Liste des années de 2010 à 2024
annees = range(10, 25)

# Loop pour renommer les colonnes
for i in annees:
    col_budget_old = f"('budget', 20{i})"
    col_budget_new = f"budget_{i}"
    col_prime_old = f"('prime', 20{i})"
    col_prime_new = f"prime_{i}"
    col_partprime_old = f"('part_prime', 20{i})"
    col_partprime_new = f"part_prime_{i}"

    # Renommage des colonnes
    budget_pivot.rename(
        columns={
            col_budget_old: col_budget_new,
            col_prime_old: col_prime_new,
            col_partprime_old: col_partprime_new,
        },
        inplace=True,
    )

budget_pivot.columns

Index(['insee', 'nom_com', 'catnat_count', 'budget_10', 'budget_11',
       'budget_12', 'budget_13', 'budget_14', 'budget_15', 'budget_16',
       'budget_17', 'budget_18', 'budget_19', 'budget_20', 'budget_21',
       'budget_22', 'budget_23', 'budget_24', 'part_prime_10', 'part_prime_11',
       'part_prime_12', 'part_prime_13', 'part_prime_14', 'part_prime_15',
       'part_prime_16', 'part_prime_17', 'part_prime_18', 'part_prime_19',
       'part_prime_20', 'part_prime_21', 'part_prime_22', 'part_prime_23',
       'part_prime_24', 'prime_10', 'prime_11', 'prime_12', 'prime_13',
       'prime_14', 'prime_15', 'prime_16', 'prime_17', 'prime_18', 'prime_19',
       'prime_20', 'prime_21', 'prime_22', 'prime_23', 'prime_24'],
      dtype='object')

##### Calcul de l'évolution de la part de la prime


In [167]:
conditions1 = [
    (budget_pivot["part_prime_15"] > 0.005) & (budget_pivot["part_prime_24"] > 0.005),
    (budget_pivot["part_prime_16"] > 0.005) & (budget_pivot["part_prime_24"] > 0.005),
    (budget_pivot["part_prime_17"] > 0.005) & (budget_pivot["part_prime_24"] > 0.005),
]

results1 = ["2015 - 2024", "2016 - 2024", "2017 - 2024"]

budget_pivot["evolution_period"] = np.select(conditions1, results1, "NA")

conditions2 = [
    (budget_pivot["evolution_period"] == "2015 - 2024"),
    (budget_pivot["evolution_period"] == "2016 - 2024"),
    (budget_pivot["evolution_period"] == "2017 - 2024"),
]

results2 = [
    ((budget_pivot["prime_24"] - budget_pivot["prime_15"]) / budget_pivot["prime_15"]),
    ((budget_pivot["prime_24"] - budget_pivot["prime_16"]) / budget_pivot["prime_16"]),
    ((budget_pivot["prime_24"] - budget_pivot["prime_17"]) / budget_pivot["prime_17"]),
]

budget_pivot["prime_evolution"] = np.select(conditions2, results2, 0)
budget_pivot

,insee,nom_com,catnat_count,budget_10,budget_11,budget_12,budget_13,budget_14,budget_15,budget_16,...,prime_17,prime_18,prime_19,prime_20,prime_21,prime_22,prime_23,prime_24,evolution_period,prime_evolution
0,01004,Ambérieu-en-Bugey,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,62894.32,NaN,50126.96,84107.74,90963.56,112284.45,103667.10,NA,0.000000
1,01007,Ambronay,12.0,NaN,NaN,NaN,NaN,NaN,2315311.5,NaN,...,NaN,20060.92,NaN,21565.17,22444.04,23200.87,27830.23,29453.43,2015 - 2024,-0.594715
2,01012,Aranc,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,7560.10,7667.45,NaN,NaN,7258.11,NaN,NaN,NA,0.000000
3,01013,Arandas,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,8827.18,NaN,NaN,NaN,NaN,NaN,NaN,NA,0.000000
4,01014,Arbent,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,41924.22,41881.74,NaN,NA,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23195,97420,Sainte-Suzanne,18.0,NaN,27248250.41,NaN,NaN,31020195.85,NaN,NaN,...,NaN,120561.25,NaN,NaN,NaN,174381.74,106521.75,148993.68,NA,0.000000
23196,97421,Salazie,20.0,NaN,9395901.63,NaN,NaN,9728751.83,NaN,NaN,...,NaN,39168.08,NaN,NaN,NaN,53934.36,NaN,62548.08,NA,0.000000
23197,97422,Le Tampon,19.0,NaN,NaN,NaN,72839460.16,72238742.79,NaN,NaN,...,NaN,504119.91,NaN,NaN,NaN,319203.63,NaN,429673.10,NA,0.000000
23198,97423,Les Trois-Bassins,12.0,NaN,8808475.59,NaN,NaN,9605368.26,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,51904.04,NaN,66897.76,NA,0.000000


##### Tranches de montant du budget


In [168]:
# Liste des années de 2010 à 2024
annees = range(10, 25)

# Catégories
results = [
    "1. 0 - 100K",
    "2. 100K-200K",
    "3. 200K-500K",
    "4. 500K-1M",
    "5.>1M",
]

# Loop to create category budget columns
for i in annees:
    col_budget = f"budget_{i}"
    col_cat = f"budget_cat_{i}"

    # 1. Remplacement des NaN par 0 pour l'année en cours
    budget_pivot[col_budget] = np.nan_to_num(budget_pivot[col_budget])

    # 2. Définition des conditions dynamiques pour l'année i
    conditions = [
        (budget_pivot[col_budget] < 100000),
        (budget_pivot[col_budget] < 200000),
        (budget_pivot[col_budget] < 500000),
        (budget_pivot[col_budget] < 1000000),
        (budget_pivot[col_budget] >= 1000000),
    ]

    # 3. Création de la colonne de catégorie correspondante
    budget_pivot[col_cat] = np.select(conditions, results)

# Vérification : Affichage des premières lignes des nouvelles colonnes
cols_cat = [f"budget_cat_{i}" for i in annees]
budget_pivot.head(5)

,insee,nom_com,catnat_count,budget_10,budget_11,budget_12,budget_13,budget_14,budget_15,budget_16,...,budget_cat_15,budget_cat_16,budget_cat_17,budget_cat_18,budget_cat_19,budget_cat_20,budget_cat_21,budget_cat_22,budget_cat_23,budget_cat_24
0,01004,Ambérieu-en-Bugey,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,5.>1M,1. 0 - 100K,5.>1M,5.>1M,5.>1M,5.>1M,5.>1M
1,01007,Ambronay,12.0,0.0,0.0,0.0,0.0,0.0,2315311.5,0.0,...,5.>1M,1. 0 - 100K,1. 0 - 100K,5.>1M,1. 0 - 100K,5.>1M,5.>1M,5.>1M,5.>1M,5.>1M
2,01012,Aranc,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,3. 200K-500K,3. 200K-500K,1. 0 - 100K,1. 0 - 100K,3. 200K-500K,1. 0 - 100K,1. 0 - 100K
3,01013,Arandas,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,3. 200K-500K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K
4,01014,Arbent,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,1. 0 - 100K,5.>1M,5.>1M,1. 0 - 100K


### Visualisation


#### Tri par ordre croissant des groupements


In [169]:
# Catégories de budget
sort1 = sorted(budget_tot["budget_cat"].unique())
sort1b = sorted(budget_pivot["budget_cat_24"].unique())

# Catégories de prime
sort2 = sorted(budget_tot["prime_cat"].unique())

# Nombre de catnats
sort3 = ["Aucune", "1", "2 - 4", "5 - 9", "10+"]

#### Graphiques


##### Part des primes & catnat


In [170]:
fig = px.scatter(
    budget_tot.loc[(budget_tot["annee"] == 2024) & (budget_tot["prime_per_hab"] < 400)],
    x="catnat_count",
    y="prime_per_hab",
    color="budget_cat",
    category_orders={"budget_cat": sort1},
)

fig.update_layout(
    title="Nombre de Cat Nat et prime par habitant 2024",
    yaxis_title="Prime par habitant",
    xaxis_title="Nombre de Cat Nat",
    legend_title="Montant du budget",
)

fig.show()

In [171]:
fig = px.scatter(
    budget_tot.loc[(budget_tot["annee"] == 2024) & (budget_tot["part_prime"] < 0.1)],
    x="catnat_count",
    y="part_prime",
    color="budget_cat",
    category_orders={"budget_cat": sort1},
)

fig.update_layout(
    title="Nombre de Cat Nat et part de la prime dans le budget de 2024",
    yaxis_title="Part des primes",
    xaxis_title="Nombre de Cat Nat",
    legend_title="Montant du budget",
    yaxis_tickformat=".0%",
)

fig.show()

In [172]:
fig = px.box(
    budget_tot.loc[(budget_tot["part_prime"] < 0.1) & (budget_tot["part_prime"] > 0)],
    x="catnat_group",
    y="part_prime",
    category_orders={"catnat_group": sort3},
    labels={"catnat_group": "Nombre de Cat Nat", "part_prime": "Part des primes"},
)

fig.update_layout(yaxis_tickformat=".0%", title="Part des primes par nombre de Cat Nat")

fig.show()

##### Evolution de primes et catnat


In [173]:
fig = px.scatter(
    budget_pivot,
    x="catnat_count",
    y="prime_evolution",
    color="budget_cat_24",
    category_orders={"budget_cat_24": sort1b},
)

fig.update_layout(
    title="Nombre de Cat Nat et evolution de la prime dans le budget",
    yaxis_title="Evolution de la prime 2015 - 2024",
    xaxis_title="Nombre de Cat Nat",
)

fig.show()

In [174]:
fig = px.scatter(
    budget_pivot.loc[budget_pivot["budget_24"] < 1000000],
    x="catnat_count",
    y="prime_evolution",
    color="budget_cat_24",
    size="budget_24",
    category_orders={"budget_cat_24": sort1b},
)

fig.update_layout(
    title="Nombre de Cat Nat et evolution de la prime dans le budget",
    yaxis_title="Evolution de la prime 2015 - 2024",
    xaxis_title="Nombre de Cat Nat",
)

fig.show()

In [175]:
fig = px.box(
    budget_pivot.loc[
        (budget_pivot["catnat_count"] < 40) & (budget_pivot["prime_evolution"] < 2)
    ],
    x="catnat_count",
    y="prime_evolution",
    labels={
        "catnat_count": "Nombre de Cat Nat",
        "prime_evolution": "Evolution de la prime",
    },
)

fig.update_layout(title="Evolution de la prime par nombre de Cat Nat")

fig.show()

In [176]:
budget_pivot.groupby(["budget_cat_24"], as_index=False).agg(
    catnat_min=("catnat_count", "min"),
    catnat_max=("catnat_count", "max"),
    catnat_avg=("catnat_count", "mean"),
    budget_24=("budget_24", "sum"),
    prime_24=("prime_24", "sum"),
    prime_evol=("prime_evolution", "mean"),
    prime_15=("prime_15", "sum"),
)

,budget_cat_24,catnat_min,catnat_max,catnat_avg,budget_24,prime_24,prime_evol,prime_15
0,1. 0 - 100K,1.0,46.0,6.933675,1.618436e+07,4.792953e+05,0.000303,46856118.35
1,2. 100K-200K,1.0,23.0,6.179407,9.503433e+07,2.428675e+06,0.081390,88660.73
2,3. 200K-500K,1.0,30.0,8.016935,3.936953e+08,8.189186e+06,0.126076,653607.16
3,4. 500K-1M,1.0,36.0,9.994641,6.619338e+08,1.173150e+07,0.165829,1463703.82
4,5.>1M,1.0,72.0,15.735587,3.019619e+10,1.963145e+08,0.177225,57542207.33


In [177]:
catnat_evol = budget_pivot.groupby(["catnat_count"], as_index=False).agg(
    communes_count=("insee", "count"),
    prime_evol_min=("prime_evolution", "min"),
    prime_evol_max=("prime_evolution", "max"),
    prime_evol_mean=("prime_evolution", "mean"),
)

In [178]:
catnat = catnat_evol["catnat_count"]
commune_count = catnat_evol["communes_count"]
evol_mean = catnat_evol["prime_evol_mean"]

fig = go.Figure(data=go.Bar(x=catnat, y=commune_count, name="Nombre de Cat Nat"))

fig.add_trace(
    go.Scatter(x=catnat, y=evol_mean, yaxis="y2", name="Evolution de la prime")
)

fig.update_layout(
    title="Nombre de Cat Nat et evolution de la prime entre 2015 et 2024",
    legend=dict(orientation="h"),
    yaxis2=dict(title=dict(text="Evolution de la prime"), side="right", overlaying="y"),
)

fig.show()